In [2]:
from pyspark.sql import SparkSession

# Initialize SparkSession if not already initialized
if 'spark' not in locals() or spark is None:
    spark = SparkSession.builder \
        .appName("DataFrame Creation") \
        .getOrCreate()

data = [
    (1, "Rahul", "IT", 70000, "2024-01-10"),
    (2, "Sneha", "HR", 60000, "2024-02-15"),
    (3, "Arjun", "IT", 75000, "2024-03-01"),
    (4, "Priya", "Finance", 80000, "2024-01-25"),
    (5, "Karan", None, 50000, "2024-02-20")
]

columns = ["emp_id", "name", "department", "salary", "joining_date"]

df = spark.createDataFrame(data, columns)
df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [4]:
df.show()
df.printSchema()
df.columns
df.count()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- joining_date: string (nullable = true)



5

In [5]:
df.show(3)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
+------+-----+----------+------+------------+
only showing top 3 rows


In [6]:
df_renamed = df.withColumnRenamed("salary", "emp_salary")
df_renamed.show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2024-01-10|
|     2|Sneha|        HR|     60000|  2024-02-15|
|     3|Arjun|        IT|     75000|  2024-03-01|
|     4|Priya|   Finance|     80000|  2024-01-25|
|     5|Karan|      NULL|     50000|  2024-02-20|
+------+-----+----------+----------+------------+



In [7]:
df.filter(df.department == "IT").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     3|Arjun|        IT| 75000|  2024-03-01|
+------+-----+----------+------+------------+



In [8]:
df.filter((df.salary > 60000) & (df.department == "IT")).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     3|Arjun|        IT| 75000|  2024-03-01|
+------+-----+----------+------+------------+



In [9]:
df.drop("joining_date").show()

+------+-----+----------+------+
|emp_id| name|department|salary|
+------+-----+----------+------+
|     1|Rahul|        IT| 70000|
|     2|Sneha|        HR| 60000|
|     3|Arjun|        IT| 75000|
|     4|Priya|   Finance| 80000|
|     5|Karan|      NULL| 50000|
+------+-----+----------+------+



In [10]:
from pyspark.sql.functions import sum, avg, max, min

df.select(
    sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
).show()

+------------+----------+----------+----------+
|total_salary|avg_salary|max_salary|min_salary|
+------------+----------+----------+----------+
|      335000|   67000.0|     80000|     50000|
+------------+----------+----------+----------+



In [11]:
dept_data = [

    ("IT", "Bangalore"),

    ("HR", "Mumbai"),

    ("Finance", "Delhi")

]

dept_columns = ["department", "location"]

dept_df = spark.createDataFrame(dept_data, dept_columns)

dept_df.show()


+----------+---------+
|department| location|
+----------+---------+
|        IT|Bangalore|
|        HR|   Mumbai|
|   Finance|    Delhi|
+----------+---------+



In [12]:
df.join(dept_df, on="department", how="inner").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|   Finance|     4|Priya| 80000|  2024-01-25|    Delhi|
|        HR|     2|Sneha| 60000|  2024-02-15|   Mumbai|
|        IT|     1|Rahul| 70000|  2024-01-10|Bangalore|
|        IT|     3|Arjun| 75000|  2024-03-01|Bangalore|
+----------+------+-----+------+------------+---------+



In [13]:
df.join(dept_df, on="department", how="left").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        HR|     2|Sneha| 60000|  2024-02-15|   Mumbai|
|        IT|     1|Rahul| 70000|  2024-01-10|Bangalore|
|      NULL|     5|Karan| 50000|  2024-02-20|     NULL|
|   Finance|     4|Priya| 80000|  2024-01-25|    Delhi|
|        IT|     3|Arjun| 75000|  2024-03-01|Bangalore|
+----------+------+-----+------+------------+---------+



In [14]:
df.join(dept_df, on="department", how="right").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     3|Arjun| 75000|  2024-03-01|Bangalore|
|        IT|     1|Rahul| 70000|  2024-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2024-02-15|   Mumbai|
|   Finance|     4|Priya| 80000|  2024-01-25|    Delhi|
+----------+------+-----+------+------------+---------+



In [15]:
df.withColumn("bonus", df.salary * 0.1).show()

+------+-----+----------+------+------------+------+
|emp_id| name|department|salary|joining_date| bonus|
+------+-----+----------+------+------------+------+
|     1|Rahul|        IT| 70000|  2024-01-10|7000.0|
|     2|Sneha|        HR| 60000|  2024-02-15|6000.0|
|     3|Arjun|        IT| 75000|  2024-03-01|7500.0|
|     4|Priya|   Finance| 80000|  2024-01-25|8000.0|
|     5|Karan|      NULL| 50000|  2024-02-20|5000.0|
+------+-----+----------+------+------------+------+



In [16]:
df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [19]:
from pyspark.sql.functions import col, isnull

df.filter(isnull(col("department"))).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [21]:
df.na.fill({"department": "Unknown"}).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|   Unknown| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [22]:
df.na.drop().show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
+------+-----+----------+------+------------+



In [24]:
from pyspark.sql.functions import upper, lower, length

df.select(
    "name",
    upper("name").alias("upper_name"),
    lower("name").alias("lower_name"),
    length("name").alias("name_length")
).show()

+-----+----------+----------+-----------+
| name|upper_name|lower_name|name_length|
+-----+----------+----------+-----------+
|Rahul|     RAHUL|     rahul|          5|
|Sneha|     SNEHA|     sneha|          5|
|Arjun|     ARJUN|     arjun|          5|
|Priya|     PRIYA|     priya|          5|
|Karan|     KARAN|     karan|          5|
+-----+----------+----------+-----------+



In [27]:
from google.colab import files
uploaded = files.upload()

Saving departments.csv to departments (2).csv
Saving employees.csv to employees (2).csv
Saving employees_nested.json to employees_nested (2).json
Saving sales (1).csv to sales (1).csv


In [28]:
!ls

'departments (1).csv'  'employees (2).csv'	     employees_nested.json
'departments (2).csv'   employees.csv		    'sales (1).csv'
 departments.csv       'employees_nested (1).json'   sample_data
'employees (1).csv'    'employees_nested (2).json'


In [29]:
df = spark.read.csv("employees.csv", header = True, inferSchema=True)
df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+

